# Pipeline de forecasting — preço do petróleo Brent

Notebook obrigatório da prova substitutiva: extração, features, treino temporal, métricas e serialização do modelo campeão.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import load_or_refresh, series_summary
from src.feature_engineering import build_features, feature_columns, recursive_forecast
from src.model_trainer import save_bundle, temporal_split, train_and_select

raw = load_or_refresh(refresh=False)
display(pd.Series(series_summary(raw), name='serie'))

## Engenharia de atributos

Lags t−1, t−2, t−3, t−5, t−7, t−15, t−30; médias 7/14/30/90; volatilidade 7/30; calendário. Janelas móveis usam apenas t−1 para evitar vazamento.

In [ ]:
features = build_features(raw)
cols = feature_columns()
print('atributos:', cols)
print('shape:', features.shape)
features[['date', 'price', *cols[:6]]].tail()

## Treinamento comparativo (Time Series Split)

Últimos 60 dias úteis ficam no teste. Sem shuffle. Candidatos: Naive (lag-1), Random Forest, XGBoost; Prophet/SARIMAX/LightGBM se instalados.

In [ ]:
bundle = train_and_select(refresh=False, test_days=60)
path = save_bundle(bundle)
print('campeão:', bundle['model_name'])
print('modelo salvo em', path)
display(pd.DataFrame(bundle['comparison']))

## Métricas no conjunto de teste (1 passo à frente)

In [ ]:
display(pd.Series(bundle['metrics'], name='teste_1_passo'))
if bundle.get('horizon_metrics'):
    display(pd.DataFrame(bundle['horizon_metrics']).T.rename_axis('horizonte'))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.to_datetime(bundle['y_dates']), bundle['y_true'], label='Real', color='#0A3D62')
ax.plot(pd.to_datetime(bundle['y_dates']), bundle['y_pred'], label='Previsto', color='#C9A227', linestyle='--')
ax.set_title('Teste temporal — real vs. previsto')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Projeção recursiva a partir do último preço oficial

In [ ]:
forecast = recursive_forecast(
    bundle['model'],
    raw,
    horizon=15,
    feature_cols=bundle['feature_columns'],
    residual_std=bundle['residual_std'],
)
display(forecast)
fig, ax = plt.subplots(figsize=(12, 4))
hist = raw.tail(90)
ax.plot(hist['date'], hist['price'], label='Histórico', color='#0A3D62')
ax.plot(forecast['date'], forecast['predicted'], label='Projeção 15d', color='#C9A227', linestyle='--')
ax.fill_between(forecast['date'], forecast['lower'], forecast['upper'], color='#C9A227', alpha=0.2)
ax.set_title('Projeção recursiva — 15 dias úteis')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Governança

- Métrica principal: MAPE de curto prazo (meta ≤ 5% em 7–15 dias).
- RMSE e MAE em US$ para a mesa de trading.
- Validação em janela expansiva registrada em `bundle['rolling_cv']`.
- Artefato de produção: `app/model.joblib`.

In [ ]:
display(pd.DataFrame(bundle.get('rolling_cv', [])))
print('train_end', bundle['train_end'], 'teste', bundle['test_start'], '→', bundle['test_end'])